# Trace figure: demo on the running example

Draft for the visit conceptualisation (`overview/Single Level Visit Mappings.md`); the figure is not part of the tool yet.
The functions live in `ocelescope_module_location_based_ocpm.visits.trace_figure` (and `visits.context`); details in the docstrings (e.g. `help(trace_figure)`).

Start Jupyter from the repository root with

```
uv run --with jupyter jupyter lab notebooks/trace_figure.ipynb
```

In [ ]:
from pathlib import Path

import polars as pl

from ocelescope_module_location_based_ocpm.derivation.level_files import load_level_mapping
from ocelescope_module_location_based_ocpm.visits.context import location_context
from ocelescope_module_location_based_ocpm.visits.trace_figure import (
    add_high_level_locations,
    add_location_kinds,
    high_level_runs,
    relations_for_figure,
    trace_figure,
)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
EXAMPLE = ROOT / "data" / "running_example"

## 1. Inputs

**Location mapping.** The current locations of one level, as saved by the location derivation notebook: `data/running_example/derived/locations_level_{k}.csv`, one row per hu-event relation with `loc` and `ploc` (empty = ⊥).
`load_level_mapping(folder, level)` reads one level with the right types.

**Location context.** $agg$, $V$ and $W$ from `data/running_example/inputs/locations.csv` (one row per location; columns `location`, `high_level`, `is_vehicle`, `is_storage`).
An empty `high_level` leaves the location unmapped, so it is its own high-level location (T here).
`location_context(table)` turns the table into a `LocationContext` with `agg` (dict), `vehicles` and `storage_units` (sets).
The file format is a draft until the inputs for visits are decided.

In [ ]:
LEVEL = 3
mapping = load_level_mapping(EXAMPLE / "derived", LEVEL)
mapping

In [ ]:
locations = pl.read_csv(EXAMPLE / "inputs" / "locations.csv")
context = location_context(locations)
locations, context

## 2. Prepare the relations: `relations_for_figure`

`relations_for_figure(relations, location, handling_units, agg, vehicles, storage_units, axis)` selects the relations and adds everything the figure needs.

| parameter | meaning | default |
|---|---|---|
| `relations` | one row per hu-event relation, e.g. a loaded level mapping | |
| `location` | column holding $loc$ | `"loc"` |
| `handling_units` | handling units to show, top to bottom in this order | all, sorted by id |
| `agg` | location mapping $agg: L \not\to HL$ as a dict; locations it does not map are their own high-level location (identity) | identity for all |
| `vehicles` | the set $V$ (drawn as open circles) | empty |
| `storage_units` | the set $W$ (drawn as diamonds); a location in both $V$ and $W$ counts as a vehicle | empty |
| `axis` | x axis: `"event order"` (shared events line up), `"trace position"` (index in $otrace(hu)$, as in the visit definition) or `"time"` | `"event order"` |

In [ ]:
HUS = ["s1", "c1", "c2", "c3", "c4", "c5"]

relations = relations_for_figure(mapping, "loc", HUS, context.agg, context.vehicles, context.storage_units)
relations.filter(pl.col("hu") == "c1")

### The steps inside

`relations_for_figure` only chains three small functions, which also work on any frame with the right columns:
- `add_high_level_locations(frame, location, agg)`: column `high_level` $= \widehat{hl}$ (⊥ stays ⊥),
- `add_location_kinds(frame, location, vehicles, storage_units)`: column `location_kind` (`vehicle`, `storage`, `other`; ⊥ for ⊥),
- `add_x_positions(frame, axis)`: columns `position` (trace position) and `x`.

`high_level_runs(relations)` gives the runs of consecutive relations with the same high-level location (used by the ribbons).

In [ ]:
toy = pl.DataFrame({"loc": ["A1", "A2", "B1", "T", None]})
toy = add_high_level_locations(toy, "loc", context.agg)
add_location_kinds(toy, "loc", context.vehicles, context.storage_units)

In [ ]:
high_level_runs(relations.filter(pl.col("hu") == "c1"))

## 3. The figure: `trace_figure`

| parameter | meaning | default |
|---|---|---|
| `relations` | output of `relations_for_figure` | |
| `visits` | optional table of visits, one row per visit (section 5) | none |
| `filled_groups` | visit groups drawn as filled bars; all others outlined | `("resolved",)` |
| `high_level_style` | how $\widehat{hl}$ is drawn: `"ribbon"`, one labelled bar per run of the same high-level location below the trace (the other values are earlier alternatives) | `"ribbon"` |
| `show_event_labels` | event ids at an angle above the markers | `True` |
| `link_shared_events` | dashed lines joining an event shared by several handling units (ignored on the trace position axis) | `True` |
| `title` | figure title | none |
| `width` | figure width in pixels; a ribbon label is only shown if it fits at this width (otherwise legend and hover name the location) | 1200 |

Hover over a marker or ribbon for the event, activity, time, trace position and both locations.

In [ ]:
trace_figure(relations, title=f"Running example, Level {LEVEL}").show()

## 4. Axes

- `"event order"`: rank of the event among all shown events; an event shared by several handling units has one x, and dashed lines join them.
- `"trace position"`: the index in each handling unit's own trace, so shared events have different x values (no dashed lines). This is the position the visit definition counts in.
- `"time"`: real time; gaps between events become visible.

In [ ]:
for axis in ("trace position", "time"):
    rel = relations_for_figure(mapping, "loc", HUS, context.agg, context.vehicles, context.storage_units, axis=axis)
    trace_figure(rel, title=f"x = {axis}").show()

## 5. Visits

Visits are not computed yet. `visits` takes a table with the columns `hu`, `high_level` (location of the visit), `first_event`, `last_event` and `group` (e.g. "maximal", "resolved").
Each group gets its own block of lanes below the handling unit, labelled with the group name; visits sharing relations go into different lanes (`visit_lanes`), so contained and overlapping visits are visible.

Below: the maximal visits at κ = 2, written by hand for illustration.

In [ ]:
VISITS_K2 = pl.DataFrame(
    [
        ("s1", "A", "e1", "e3"),
        ("c1", "A", "e3", "e11"),
        ("c1", "B", "e7", "e7"),
        ("c1", "T", "e8", "e14"),
        ("c2", "A", "e3", "e11"),
        ("c2", "T", "e8", "e8"),
        ("c2", "B", "e12", "e15"),
        ("c3", "T", "e8", "e8"),
        ("c3", "B", "e10", "e14"),
        ("c4", "B", "e15", "e16"),
    ],
    schema=["hu", "high_level", "first_event", "last_event"],
    orient="row",
).with_columns(pl.lit("maximal").alias("group"))

trace_figure(relations, VISITS_K2, title="Maximal visits at κ = 2 (illustration)").show()

`filled_groups` decides which groups are filled; here the same visits as a filled group, as resolved visits will look later.

In [ ]:
trace_figure(relations, VISITS_K2, filled_groups=("maximal",), title="Filled visit bars").show()

## 6. Saving

`fig.write_html(path)` saves an interactive file (`include_plotlyjs="cdn"` gives a small file that needs internet to open).

In [ ]:
# trace_figure(relations, VISITS_K2).write_html(ROOT / "notebooks" / "trace-figure.html")